# BioVision AI — Train real disease-classification models in Google Colab

Run this notebook on your phone or any browser; a laptop is not required. It downloads real Kaggle datasets, trains the repository's MobileNetV2 models, and packages the resulting model weights and class maps.

**Security:** The Kaggle token previously pasted into ChatGPT should be revoked immediately. Create a fresh token and enter it only in the password prompt below. Never paste tokens into chat, notebook source, GitHub, or screenshots. Human skin-lesion data is non-commercial and this model is not a medical diagnostic tool.

**Compute/time:** Training all three domains can take a long time and may exceed a free Colab session. If it stops, change `DOMAINS` to just `['plant']` and run one domain at a time. Review each dataset's license and labels before using it.

In [ ]:
# Install dependencies (Colab's Python version may differ from local requirements)
%pip -q install kaggle streamlit opencv-python-headless pillow pandas scikit-learn matplotlib plotly seaborn reportlab pyttsx3 tqdm scipy
%pip -q install --upgrade tensorflow


In [ ]:
# Enter a NEW Kaggle API token privately. Input is hidden and not written to this notebook.
import os, getpass
token = getpass.getpass('Paste your NEW Kaggle API token (input hidden): ').strip()
if not token.startswith('KGAT_'):
    raise ValueError('This notebook expects a Kaggle API token beginning with KGAT_. Generate a new token in Kaggle Settings > API.')
os.environ['KAGGLE_API_TOKEN'] = token
del token
print('Token set for this runtime; it was not saved into the notebook.')


In [ ]:
# Get the project source
%cd /content
!rm -rf AI-Multi-Domain-Disease-Prediction
!git clone --depth 1 https://github.com/247r1a05c2-b/AI-Multi-Domain-Disease-Prediction.git
%cd /content/AI-Multi-Domain-Disease-Prediction/BioVisionAI
print('Project ready:', os.getcwd())


## Choose domains and train

Default below attempts all three domains. If the session runs out of time or storage, run one domain at a time. Dataset availability, Kaggle access, licensing, and Colab resource limits can affect completion.

In [ ]:
import subprocess, sys
DOMAINS = ['plant', 'human', 'animal']  # or ['plant'] to start with the smaller first task
cmd = [sys.executable, 'training/prepare_real_datasets.py', '--domains', *DOMAINS, '--train']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Verify artifacts exist and package model weights + class maps
from pathlib import Path
import zipfile
model_dir = Path('saved_models')
artifacts = sorted(p for p in model_dir.glob('*') if p.is_file() and (p.suffix in {'.h5', '.keras', '.json'}))
for p in artifacts:
    print(p.name, f'{p.stat().st_size / (1024*1024):.1f} MiB')
if not any(p.suffix in {'.h5', '.keras'} for p in artifacts):
    raise RuntimeError('No trained model file found; do not treat this run as successful.')
zip_path = '/content/biovision_trained_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        z.write(p, arcname=p.name)
print('Created:', zip_path)
from google.colab import files
files.download(zip_path)


## Important deployment note

This ZIP contains trained artifacts; it does **not** automatically upload them to GitHub or deploy them to Streamlit. Large model files may exceed GitHub's normal file-size limits. Do not commit Kaggle credentials or dataset images. Before public deployment, host model artifacts securely (for example, in an appropriate model repository) and configure the app to download the exact version; test predictions and label mappings first. Human skin-lesion predictions are educational only and must not be used for diagnosis.